# 🌤️ Weather Prediction v3 - PREDICTIONS ONLY

## ⚡ Quick Predictions with Pre-Trained Models

**Use this notebook when:**
- ✅ You've already trained v3 models
- ✅ You just want to make predictions
- ✅ You don't want to retrain (saves hours!)

**Execution time:** ~30 seconds per prediction

---

## Setup (Run Once)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("✓ Drive mounted!")

Mounted at /content/drive
✓ Drive mounted!


In [2]:
# Install only what we need for predictions
!pip install -q lightgbm xgboost stable-baselines3 gymnasium torch --quiet

print("✓ Packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 4.4 MB/s eta 0:00:00
✓ Packages installed!


In [3]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import pickle
import warnings
warnings.filterwarnings('ignore')

from stable_baselines3 import PPO

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_PATH = '/content/drive/MyDrive/FYP_WeatherPrediction'

# Use v3 models and data
DATA_PATH = f'{BASE_PATH}/data/pak_weather_engineered_v3.csv'
MODELS_DIR = f'{BASE_PATH}/models_city_v3'
PREDICTIONS_LOG = f'{BASE_PATH}/predictions_v3/predictions_log.csv'

CITIES = {
    'Karachi': {'lat': 24.8607, 'lon': 67.0011},
    'Lahore': {'lat': 31.5204, 'lon': 74.3587},
    'Islamabad': {'lat': 33.6844, 'lon': 73.0479},
    'Peshawar': {'lat': 34.0151, 'lon': 71.5249},
    'Quetta': {'lat': 30.1798, 'lon': 66.9750}
}

FEATURES = ['temperature_max', 'temperature_min', 'precipitation', 'wind_speed']

# Create predictions directory
os.makedirs(f'{BASE_PATH}/predictions_v3', exist_ok=True)

print("✓ Configuration loaded")
print(f"  Models: {MODELS_DIR}")
print(f"  Data: {DATA_PATH}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✓ Configuration loaded
  Models: /content/drive/MyDrive/FYP_WeatherPrediction/models_city_v3
  Data: /content/drive/MyDrive/FYP_WeatherPrediction/data/pak_weather_engineered_v3.csv


In [4]:
# ============================================================================
# VERIFY MODELS EXIST
# ============================================================================

print("\nChecking for trained v3 models...\n")

models_found = True

for city in CITIES.keys():
    city_dir = f"{MODELS_DIR}/{city}"

    if not os.path.exists(city_dir):
        print(f"❌ {city}: Models not found")
        models_found = False
    else:
        # Check for at least temperature model
        temp_model = f"{city_dir}/temp_max_lgb.pkl"
        if os.path.exists(temp_model):
            print(f"✓ {city}: Models ready")
        else:
            print(f"⚠️  {city}: Incomplete models")
            models_found = False

if models_found:
    print("\n✅ All v3 models found! Ready for predictions.")
else:
    print("\n❌ ERROR: v3 models not found!")
    print("\nPlease run the complete training notebook first:")
    print("  Predict_v3_Complete_Pipeline.ipynb")
    raise FileNotFoundError("v3 models not trained yet")


Checking for trained v3 models...

✓ Karachi: Models ready
✓ Lahore: Models ready
✓ Islamabad: Models ready
✓ Peshawar: Models ready
✓ Quetta: Models ready

✅ All v3 models found! Ready for predictions.


## Prediction Functions

In [5]:
def create_features_for_prediction(df, city, pred_date, target_feature):
    """
    Create features for prediction date
    """
    city_df = df[df['city'] == city].copy()
    city_df = city_df.sort_values('date').reset_index(drop=True)

    # Get latest row as template
    latest_row = city_df.iloc[-1]
    features = latest_row.to_dict()

    # Update time-based features for prediction date
    features['year'] = pred_date.year
    features['month'] = pred_date.month
    features['day'] = pred_date.day
    features['day_of_year'] = pred_date.timetuple().tm_yday
    features['week'] = pred_date.isocalendar()[1]
    features['day_of_week'] = pred_date.weekday()
    features['is_weekend'] = 1 if pred_date.weekday() >= 5 else 0
    features['is_month_start'] = 1 if pred_date.day == 1 else 0
    features['is_month_end'] = 1 if pred_date.day >= 28 else 0
    features['quarter'] = (pred_date.month - 1) // 3 + 1

    # Cyclical features
    features['sin_month'] = np.sin(2 * np.pi * pred_date.month / 12)
    features['cos_month'] = np.cos(2 * np.pi * pred_date.month / 12)
    features['sin_day_of_year'] = np.sin(2 * np.pi * features['day_of_year'] / 365)
    features['cos_day_of_year'] = np.cos(2 * np.pi * features['day_of_year'] / 365)
    features['sin_week'] = np.sin(2 * np.pi * features['week'] / 52)
    features['cos_week'] = np.cos(2 * np.pi * features['week'] / 52)
    features['sin_day_of_week'] = np.sin(2 * np.pi * features['day_of_week'] / 7)
    features['cos_day_of_week'] = np.cos(2 * np.pi * features['day_of_week'] / 7)

    # Seasonal features
    month = pred_date.month
    if month in [12, 1, 2]:
        season = 0
    elif month in [3, 4, 5]:
        season = 1
    elif month in [6, 7, 8, 9]:
        season = 2
    else:
        season = 3

    features['season'] = season
    features['is_winter'] = 1 if season == 0 else 0
    features['is_spring'] = 1 if season == 1 else 0
    features['is_summer'] = 1 if season == 2 else 0
    features['is_monsoon'] = 1 if month in [6, 7, 8, 9] else 0

    return features

def load_model_and_predict(city, target_feature, features):
    """
    Load baseline model and predict
    """
    # Determine model file
    if 'max' in target_feature:
        model_file = 'temp_max_lgb.pkl'
    elif 'min' in target_feature:
        model_file = 'temp_min_lgb.pkl'
    elif 'precip' in target_feature:
        model_file = 'prec_lgb.pkl'
    else:
        model_file = 'wind_xgb.pkl'

    model_path = f"{MODELS_DIR}/{city}/{model_file}"

    with open(model_path, 'rb') as f:
        model_data = pickle.load(f)

    # Extract model and feature names
    if isinstance(model_data, dict):
        model = model_data.get('model', model_data)
        feature_names = model_data.get('feature_names', [])
    else:
        model = model_data
        feature_names = model.feature_name_ if hasattr(model, 'feature_name_') else []

    # Prepare feature vector
    X = [features.get(fname, 0) for fname in feature_names]
    X = np.array(X).reshape(1, -1)

    baseline_pred = model.predict(X)[0]

    return baseline_pred, feature_names

def apply_rl_correction(city, baseline_pred, features, target_feature):
    """
    Apply RL correction to baseline prediction
    """
    rl_path = f"{MODELS_DIR}/{city}/rl/ppo_agent_v2.zip"

    if not os.path.exists(rl_path):
        return baseline_pred, 0.0

    try:
        model = PPO.load(rl_path)
        obs_shape = model.observation_space.shape[0]

        if obs_shape == 15:
            # v2/v3 model (15 features)
            observation = np.array([
                baseline_pred,
                features.get(f'{target_feature}_roll_mean_7', baseline_pred),
                features.get(f'{target_feature}_roll_std_7', 0),
                features.get(f'{target_feature}_climatology', baseline_pred),
                features.get('month', 1),
                features.get(f'{target_feature}_diff_7', 0),
                features.get(f'{target_feature}_anomaly', 0),
                features.get('days_since_rain', 0),
                features.get('wind_speed_persistence', 0),
                features.get(f'{target_feature}_volatility_7', 0),
                features.get('season', 0),
                features.get('temp_above_ma7', 0),
                features.get(f'{target_feature}_roll_range_7', 0),
                features.get('precipitation_roll_sum_7', 0),
                features.get('is_monsoon', 0),
            ], dtype=np.float32)
        else:
            # v1 model (3 features)
            observation = np.array([
                baseline_pred,
                features.get(f'{target_feature}_persistence', baseline_pred),
                features.get(f'{target_feature}_climatology', baseline_pred)
            ], dtype=np.float32)

        # Normalize observation
        obs_min = observation.min()
        obs_max = observation.max()
        obs_range = obs_max - obs_min + 1e-8
        observation_norm = (observation - obs_min) / obs_range
        observation_norm = np.clip(observation_norm, 0, 1)

        # Get RL correction
        action, _ = model.predict(observation_norm, deterministic=True)
        correction = float(np.asarray(action).ravel()[0])

        rl_pred = baseline_pred + correction

        return rl_pred, correction

    except Exception as e:
        print(f"  ⚠️  RL correction failed: {e}")
        return baseline_pred, 0.0

def save_prediction(city, target_feature, pred_date, baseline_pred, rl_pred, correction):
    """
    Save prediction to log file
    """
    pred_log = pd.DataFrame([{
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'city': city,
        'feature': target_feature,
        'prediction_date': pred_date.strftime('%Y-%m-%d'),
        'baseline_prediction': round(baseline_pred, 2),
        'rl_correction': round(correction, 2),
        'rl_prediction': round(rl_pred, 2),
        'model_version': 'v3'
    }])

    if os.path.exists(PREDICTIONS_LOG):
        pred_log.to_csv(PREDICTIONS_LOG, mode='a', header=False, index=False)
    else:
        pred_log.to_csv(PREDICTIONS_LOG, index=False)

print("✓ Prediction functions loaded!")

✓ Prediction functions loaded!


## Load Dataset (One Time)

In [6]:
print("Loading engineered dataset...")
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

latest_date = df['date'].max()

print(f"✓ Dataset loaded")
print(f"  Shape: {df.shape}")
print(f"  Latest date: {latest_date.strftime('%Y-%m-%d')}")
print(f"  Cities: {', '.join(df['city'].unique())}")
print(f"\n✅ Ready to make predictions for dates after {latest_date.strftime('%Y-%m-%d')}")

Loading engineered dataset...
✓ Dataset loaded
  Shape: (12785, 108)
  Latest date: 2025-12-31
  Cities: Islamabad, Karachi, Lahore, Peshawar, Quetta

✅ Ready to make predictions for dates after 2025-12-31


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
# 🎯 MAKE PREDICTION

**Run this cell as many times as you want!**

Each run asks for: City, Feature, Date

---

In [7]:
print("\n" + "="*70)
print("WEATHER PREDICTION (Using v3 Models)")
print("="*70)

# ============================================================================
# USER INPUT
# ============================================================================

print("\nAvailable cities:")
for i, city in enumerate(CITIES.keys(), 1):
    print(f"  {i}. {city}")

city_idx = int(input("\nSelect city (1-5): ")) - 1
city = list(CITIES.keys())[city_idx]

print("\nAvailable features:")
for i, feature in enumerate(FEATURES, 1):
    print(f"  {i}. {feature}")

feature_idx = int(input("\nSelect feature (1-4): ")) - 1
target_feature = FEATURES[feature_idx]

date_str = input("\nEnter prediction date (YYYY-MM-DD): ")
pred_date = datetime.strptime(date_str, '%Y-%m-%d')

# ============================================================================
# MAKE PREDICTION
# ============================================================================

print("\n" + "="*70)
print(f"PREDICTING: {target_feature} for {city} on {pred_date.strftime('%Y-%m-%d')}")
print("="*70)

print("\n⏳ Creating features...")
features = create_features_for_prediction(df, city, pred_date, target_feature)

print("⏳ Loading baseline model...")
baseline_pred, feature_names = load_model_and_predict(city, target_feature, features)

print("⏳ Applying RL correction...")
rl_pred, correction = apply_rl_correction(city, baseline_pred, features, target_feature)

# ============================================================================
# DISPLAY RESULTS
# ============================================================================

print("\n" + "="*70)
print("PREDICTION RESULTS")
print("="*70)
print(f"\n📍 City:          {city}")
print(f"🌡️  Feature:       {target_feature}")
print(f"📅 Date:          {pred_date.strftime('%Y-%m-%d')}")
print(f"🤖 Model:         v3 (Trained on 2019-2025)")
print(f"\n{'─'*70}")
print(f"Baseline:      {baseline_pred:.2f}")
print(f"RL Correction: {correction:+.2f}")
print(f"Final (RL):    {rl_pred:.2f}")
print("="*70)

# ============================================================================
# SAVE PREDICTION
# ============================================================================

save_prediction(city, target_feature, pred_date, baseline_pred, rl_pred, correction)
print(f"\n✓ Prediction saved to: {PREDICTIONS_LOG}")

print("\n💡 Run this cell again to make another prediction!")


WEATHER PREDICTION (Using v3 Models)

Available cities:
  1. Karachi
  2. Lahore
  3. Islamabad
  4. Peshawar
  5. Quetta

Select city (1-5): 1

Available features:
  1. temperature_max
  2. temperature_min
  3. precipitation
  4. wind_speed

Select feature (1-4): 1

Enter prediction date (YYYY-MM-DD): 2026-04-01

PREDICTING: temperature_max for Karachi on 2026-04-01

⏳ Creating features...
⏳ Loading baseline model...
⏳ Applying RL correction...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



PREDICTION RESULTS

📍 City:          Karachi
🌡️  Feature:       temperature_max
📅 Date:          2026-04-01
🤖 Model:         v3 (Trained on 2019-2025)

──────────────────────────────────────────────────────────────────────
Baseline:      24.33
RL Correction: +0.04
Final (RL):    24.37

✓ Prediction saved to: /content/drive/MyDrive/FYP_WeatherPrediction/predictions_v3/predictions_log.csv

💡 Run this cell again to make another prediction!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
# 📊 VIEW PREDICTION HISTORY

See all predictions you've made

---

In [8]:
if os.path.exists(PREDICTIONS_LOG):
    pred_history = pd.read_csv(PREDICTIONS_LOG)

    print("\n" + "="*70)
    print("PREDICTION HISTORY")
    print("="*70)

    print(f"\nTotal predictions: {len(pred_history)}")

    # Show last 10
    print("\nLast 10 predictions:")
    print(pred_history.tail(10).to_string(index=False))

    # Summary by city
    print("\n" + "─"*70)
    print("Predictions by city:")
    print(pred_history['city'].value_counts())

    # Summary by feature
    print("\n" + "─"*70)
    print("Predictions by feature:")
    print(pred_history['feature'].value_counts())

else:
    print("No predictions made yet. Run the prediction cell above!")


PREDICTION HISTORY

Total predictions: 4

Last 10 predictions:
          timestamp    city         feature prediction_date  baseline_prediction  rl_correction  rl_prediction model_version
2026-04-11 19:26:54  Lahore temperature_max      2026-04-12                19.78          -0.03          19.75            v3
2026-04-11 19:28:22 Karachi temperature_min      2026-01-01                18.72           0.05          18.77            v3
2026-04-11 19:32:29 Karachi temperature_min      2026-01-01                18.72           0.05          18.77            v3
2026-04-12 06:13:56 Karachi temperature_max      2026-04-01                24.33           0.04          24.37            v3

──────────────────────────────────────────────────────────────────────
Predictions by city:
city
Karachi    3
Lahore     1
Name: count, dtype: int64

──────────────────────────────────────────────────────────────────────
Predictions by feature:
feature
temperature_max    2
temperature_min    2
Name: count, dt

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
# 🔄 BATCH PREDICTIONS

Predict multiple dates at once (Optional)

---

In [9]:
print("\n" + "="*70)
print("BATCH PREDICTIONS")
print("="*70)

# Example: Predict temperature_max for all cities for next 7 days
city_choice = input("\nSelect city (or 'all' for all cities): ")
if city_choice.lower() == 'all':
    cities_to_predict = list(CITIES.keys())
else:
    cities_to_predict = [city_choice]

feature_choice = input("Select feature (temperature_max/temperature_min/precipitation/wind_speed): ")

start_date_str = input("Start date (YYYY-MM-DD): ")
end_date_str = input("End date (YYYY-MM-DD): ")

start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
end_date = datetime.strptime(end_date_str, '%Y-%m-%d')

print(f"\nGenerating predictions...")

batch_predictions = []
current_date = start_date

while current_date <= end_date:
    for city in cities_to_predict:
        features = create_features_for_prediction(df, city, current_date, feature_choice)
        baseline_pred, _ = load_model_and_predict(city, feature_choice, features)
        rl_pred, correction = apply_rl_correction(city, baseline_pred, features, feature_choice)

        batch_predictions.append({
            'date': current_date.strftime('%Y-%m-%d'),
            'city': city,
            'feature': feature_choice,
            'baseline': round(baseline_pred, 2),
            'rl_prediction': round(rl_pred, 2),
            'correction': round(correction, 2)
        })

    current_date += pd.Timedelta(days=1)

df_batch = pd.DataFrame(batch_predictions)

print(f"\n✓ Generated {len(df_batch)} predictions")
print("\nSample:")
print(df_batch.head(10))

# Save batch predictions
batch_file = f"{BASE_PATH}/predictions_v3/batch_predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_batch.to_csv(batch_file, index=False)

print(f"\n✓ Batch predictions saved to: {batch_file}")


BATCH PREDICTIONS

Select city (or 'all' for all cities): all
Select feature (temperature_max/temperature_min/precipitation/wind_speed): temperature_max
Start date (YYYY-MM-DD): 2025-12-01
End date (YYYY-MM-DD): 2025-12-31

Generating predictions...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



✓ Generated 155 predictions

Sample:
         date       city          feature  baseline  rl_prediction  correction
0  2025-12-01    Karachi  temperature_max     24.33          24.36        0.03
1  2025-12-01     Lahore  temperature_max     19.77          19.85        0.08
2  2025-12-01  Islamabad  temperature_max     14.04          14.06        0.02
3  2025-12-01   Peshawar  temperature_max     15.46          15.67        0.21
4  2025-12-01     Quetta  temperature_max      7.95           8.05        0.10
5  2025-12-02    Karachi  temperature_max     24.33          24.36        0.03
6  2025-12-02     Lahore  temperature_max     19.79          19.87        0.08
7  2025-12-02  Islamabad  temperature_max     14.07          14.09        0.02
8  2025-12-02   Peshawar  temperature_max     15.45          15.67        0.21
9  2025-12-02     Quetta  temperature_max      7.95           8.05        0.10

✓ Batch predictions saved to: /content/drive/MyDrive/FYP_WeatherPrediction/predictions_v3/ba

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
